# LLM Judge — Google Colab GPU minimal notebook

This notebook only runs the Gemma LLM Judge over `data/candidate_examples.csv` and writes `data/candidate_examples_llm_labeled.csv`.

Run the full data pipeline locally first to create `candidate_examples.csv`, then upload or sync that `data/` folder to Google Drive. After this notebook finishes, copy the labeled CSV back into the local project and continue the local pipeline/split/model notebooks.

In Colab, select **Runtime → Change runtime type → GPU** before running.


## 0. Colab setup, Drive, and packages

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def running_in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


IN_COLAB = running_in_colab()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

PROJECT_NAME = "fandom_kg_project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive") / PROJECT_NAME
else:
    PROJECT_DIR = Path.cwd()

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Running in Colab: {IN_COLAB}")
print(f"Project directory: {PROJECT_DIR}")


def pip_install(packages: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])


for package in ["pandas", "transformers", "accelerate", "huggingface_hub", "sentencepiece", "safetensors"]:
    import_name = "huggingface_hub" if package == "huggingface_hub" else package.replace("-", "_")
    if package == "pandas":
        import_name = "pandas"
    if importlib.util.find_spec(import_name) is None or IN_COLAB:
        pip_install([package])

import torch
GPU_AVAILABLE = torch.cuda.is_available()
GPU_DEVICE = "cuda" if GPU_AVAILABLE else "cpu"
print(f"CUDA available: {GPU_AVAILABLE}")
if GPU_AVAILABLE:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. In Colab, use Runtime -> Change runtime type -> GPU.")


## 1. Configuration

In [ ]:
from pathlib import Path

DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_EXAMPLES_CSV = DATA_DIR / "candidate_examples.csv"
LLM_JUDGE_OUTPUT_CSV = DATA_DIR / "candidate_examples_llm_labeled.csv"
TRAINING_LABEL_DISTRIBUTION_CSV = DATA_DIR / "training_label_distribution.csv"

# Keep this label list consistent with the rest of the project notebooks.
RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]
NO_RELATION_LABEL = "no_relation"

USE_LLM_JUDGE = True
REQUIRE_GPU_FOR_LLM_JUDGE = True
LLM_JUDGE_MODEL_ID = "google/gemma-2-2b-it"
LLM_JUDGE_OVERWRITE_CACHE = False
LLM_JUDGE_DEVICE_MAP = "auto" if GPU_AVAILABLE else None
LLM_JUDGE_TORCH_DTYPE = "float16" if GPU_AVAILABLE else "float32"
LLM_JUDGE_MAX_NEW_TOKENS = 12
LLM_JUDGE_MAX_INPUT_CHARS = 1800
LLM_JUDGE_SAVE_EVERY = 25

print(f"LLM Judge input: {CANDIDATE_EXAMPLES_CSV}")
print(f"LLM Judge output: {LLM_JUDGE_OUTPUT_CSV}")
print(f"Input exists: {CANDIDATE_EXAMPLES_CSV.exists()}")
print(f"GPU device: {GPU_DEVICE}")


## 2. LLM Judge functions

In [ ]:
import pandas as pd
import re
from pathlib import Path

def validate_candidate_training_frame(df: pd.DataFrame, frame_name: str = "candidates_df") -> pd.DataFrame:
    """Validate the candidate DataFrame used by the downstream training pipeline."""
    required_cols = {
        "candidate_id",
        "head",
        "tail",
        "context",
        "source_type",
        "weak_label",
        "label",
        "pair_id",
        "text_basic",
        "text_marked",
    }
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(f"{frame_name} is missing required columns: {missing_cols}")

    invalid_labels = sorted(set(df["label"].dropna().astype(str)) - set(RELATIONSHIPS))
    if invalid_labels:
        raise ValueError(f"{frame_name} contains labels not in RELATIONSHIPS: {invalid_labels}")

    if df["label"].isna().any():
        raise ValueError(f"{frame_name} contains missing labels.")

    return df.reset_index(drop=True).copy()


RELATION_LABEL_DESCRIPTIONS = {
    "family": "family or relative relationship, including parent, child, sibling, spouse's family, adoptive family, or blood relation",
    "romantic": "romantic relationship, marriage, lover, fiance, spouse, betrothal, or romantic interest",
    "friend_ally": "friendship, ally, trusted companion, teammate, or cooperative relationship",
    "service_retainer": "service, retainer, attendant, guard knight, scholar, subordinate, master-servant, or duty-based relationship",
    "enemy_rival": "enemy, rival, antagonist, betrayal, hostility, violence, hatred, or opposition",
    "no_relation": "the context does not explicitly state one of the listed relationships between head and tail",
}


def build_llm_judge_prompt(row: pd.Series) -> str:
    """Build one deterministic classification prompt for a candidate pair."""
    labels_block = "\n".join(
        f"- {label}: {RELATION_LABEL_DESCRIPTIONS.get(label, label)}"
        for label in RELATIONSHIPS
    )

    context = str(row.get("context", ""))
    if len(context) > LLM_JUDGE_MAX_INPUT_CHARS:
        context = context[:LLM_JUDGE_MAX_INPUT_CHARS].rstrip() + " ..."

    section = str(row.get("section", ""))
    source_type = str(row.get("source_type", ""))
    weak_label_source = str(row.get("weak_label_source", ""))

    return f"""
You are an NLP annotation judge for a character relationship extraction dataset.

Choose exactly one relationship label from this list:
{labels_block}

Decision rules:
- Use only the supplied context, page/section metadata, head character, and tail character.
- Label the relationship from HEAD to TAIL when the context explicitly supports it.
- If the context mentions both characters but does not clearly express one listed relationship, choose no_relation.
- If more than one relationship is possible, choose the most explicit relationship in the context.
- Return only the label string. Do not explain your decision.

HEAD: {row.get("head", "")}
TAIL: {row.get("tail", "")}
SECTION: {section}
SOURCE_TYPE: {source_type}
WEAK_LABEL_SOURCE: {weak_label_source}
CONTEXT:
{context}

LABEL:
""".strip()


def parse_llm_judge_label(raw_response: str, valid_labels: list[str]) -> str | None:
    """Parse the judge response into one allowed label, or return None if parsing fails."""
    if raw_response is None:
        return None

    text = str(raw_response).strip()
    text = re.sub(r"```(?:json|text)?", "", text, flags=re.IGNORECASE).replace("```", "").strip()

    # Exact normalized match first.
    normalized = text.strip().strip('"\'`.,:;()[]{}').casefold()
    label_lookup = {label.casefold(): label for label in valid_labels}
    if normalized in label_lookup:
        return label_lookup[normalized]

    # JSON-ish or "label: X" response.
    label_pattern = r"(?:label|relationship)\s*[\"']?\s*[:=]\s*[\"']?([A-Za-z_]+)"
    match = re.search(label_pattern, text, flags=re.IGNORECASE)
    if match:
        candidate = match.group(1).casefold()
        if candidate in label_lookup:
            return label_lookup[candidate]

    # Last-resort boundary match. Sort by length so no_relation is not partially shadowed.
    for label in sorted(valid_labels, key=len, reverse=True):
        if re.search(rf"(?<![A-Za-z_]){re.escape(label)}(?![A-Za-z_])", text, flags=re.IGNORECASE):
            return label

    return None


def resolve_llm_torch_dtype(dtype_name: str):
    """Convert the config string into a torch dtype understood by transformers."""
    import torch

    if dtype_name is None or str(dtype_name).lower() == "none":
        return None
    dtype_name = str(dtype_name).lower()
    if dtype_name == "auto":
        return "auto"

    dtype_map = {
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float16": torch.float16,
        "fp16": torch.float16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }
    if dtype_name not in dtype_map:
        raise ValueError(f"Unsupported LLM_JUDGE_TORCH_DTYPE: {dtype_name}")
    return dtype_map[dtype_name]


def load_llm_judge_model(model_id: str):
    """Load the Gemma judge lazily so disabled weak-label runs do not need transformers."""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_kwargs = {"low_cpu_mem_usage": True}
    dtype = resolve_llm_torch_dtype(LLM_JUDGE_TORCH_DTYPE)
    if dtype is not None:
        model_kwargs["torch_dtype"] = dtype
    if LLM_JUDGE_DEVICE_MAP is not None:
        model_kwargs["device_map"] = LLM_JUDGE_DEVICE_MAP

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer, model


def generate_llm_judge_response(tokenizer, model, prompt: str) -> str:
    """Generate one short deterministic label response from the LLM judge."""
    import torch

    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    model_device = next(model.parameters()).device
    model_inputs = {key: value.to(model_device) for key, value in model_inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=LLM_JUDGE_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = model_inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0][prompt_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()




def ensure_huggingface_auth() -> None:
    """Configure Hugging Face authentication for gated Gemma models.

    Preferred in Colab: add a Secret named HF_TOKEN via the key icon in the left sidebar.
    Fallbacks: huggingface_hub.notebook_login(), then getpass().
    """
    import os

    existing_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
    if existing_token:
        os.environ.setdefault("HF_TOKEN", existing_token)
        os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", existing_token)
        return

    token = None
    if globals().get("IN_COLAB", False):
        try:
            from google.colab import userdata

            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None

    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = token
        return

    try:
        from huggingface_hub import notebook_login

        print("No HF_TOKEN secret found. Opening Hugging Face notebook login.")
        notebook_login()
        return
    except Exception:
        from getpass import getpass

        token = getpass("Paste your Hugging Face token: ")
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_HUB_TOKEN"] = token

def label_candidate_examples_with_llm(
    input_csv: Path,
    output_csv: Path,
    model_id: str,
) -> pd.DataFrame:
    """Read the configured candidate CSV, label every row with the LLM Judge, and save a new CSV."""

    ensure_huggingface_auth()

    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    if not input_csv.exists():
        raise FileNotFoundError(f"LLM Judge input CSV not found: {input_csv}")

    df = pd.read_csv(input_csv)
    df = validate_candidate_training_frame(df, frame_name="LLM Judge input")

    tokenizer, model = load_llm_judge_model(model_id)

    labels = []
    raw_responses = []
    parse_statuses = []
    partial_csv = output_csv.with_suffix(output_csv.suffix + ".partial")

    print(f"Running LLM Judge over {len(df):,} rows from {input_csv}")
    print(f"Model: {model_id}")

    for idx, row in df.iterrows():
        prompt = build_llm_judge_prompt(row)
        raw_response = generate_llm_judge_response(tokenizer, model, prompt)
        parsed_label = parse_llm_judge_label(raw_response, RELATIONSHIPS)

        if parsed_label is None:
            parsed_label = NO_RELATION_LABEL
            parse_status = "invalid_response_fallback_to_no_relation"
        else:
            parse_status = "parsed"

        labels.append(parsed_label)
        raw_responses.append(raw_response)
        parse_statuses.append(parse_status)

        if (idx + 1) % LLM_JUDGE_SAVE_EVERY == 0:
            partial_df = df.iloc[: idx + 1].copy()
            partial_df["label"] = labels
            partial_df["llm_judge_raw_response"] = raw_responses
            partial_df["llm_judge_parse_status"] = parse_statuses
            partial_df.to_csv(partial_csv, index=False)
            print(f"LLM Judge progress: {idx + 1:,}/{len(df):,} rows")

    df["label"] = labels
    df["llm_judge_raw_response"] = raw_responses
    df["llm_judge_parse_status"] = parse_statuses
    df["llm_judge_model"] = model_id

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)

    if partial_csv.exists():
        partial_csv.unlink()

    return validate_candidate_training_frame(df, frame_name="LLM-labeled candidates")


def load_or_create_llm_labeled_candidates(
    input_csv: Path,
    output_csv: Path,
    model_id: str,
    overwrite_cache: bool = False,
) -> pd.DataFrame:
    """Load cached LLM labels if present; otherwise run the expensive judge once."""
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    if output_csv.exists() and not overwrite_cache:
        print(f"LLM-labeled CSV already exists. Skipping LLM Judge and loading: {output_csv}")
        cached_df = pd.read_csv(output_csv)
        return validate_candidate_training_frame(cached_df, frame_name="cached LLM-labeled candidates")

    if output_csv.exists() and overwrite_cache:
        print(f"Regenerating LLM-labeled CSV because LLM_JUDGE_OVERWRITE_CACHE=True: {output_csv}")
    else:
        print(f"No LLM-labeled CSV found. Creating: {output_csv}")

    return label_candidate_examples_with_llm(input_csv=input_csv, output_csv=output_csv, model_id=model_id)


## 3. Run LLM Judge

In [ ]:
if USE_LLM_JUDGE and REQUIRE_GPU_FOR_LLM_JUDGE and not GPU_AVAILABLE:
    raise RuntimeError(
        "USE_LLM_JUDGE=True but no GPU was detected. "
        "In Colab, select Runtime -> Change runtime type -> GPU, then rerun the notebook."
    )

if not CANDIDATE_EXAMPLES_CSV.exists():
    raise FileNotFoundError(
        f"Expected input CSV not found: {CANDIDATE_EXAMPLES_CSV}. "
        "Run the local data pipeline first or upload candidate_examples.csv to the data folder."
    )

candidates_df = load_or_create_llm_labeled_candidates(
    input_csv=CANDIDATE_EXAMPLES_CSV,
    output_csv=LLM_JUDGE_OUTPUT_CSV,
    model_id=LLM_JUDGE_MODEL_ID,
    overwrite_cache=LLM_JUDGE_OVERWRITE_CACHE,
)

training_label_counts = candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count")
training_label_counts.to_csv(TRAINING_LABEL_DISTRIBUTION_CSV, index=False)

print(f"Wrote labeled candidates: {LLM_JUDGE_OUTPUT_CSV}")
print(f"Wrote label distribution: {TRAINING_LABEL_DISTRIBUTION_CSV}")
display(training_label_counts)
